# 09 · Server-side conversation state
Level L2 · Part 2 · professional · ~25 min · builds on 08

## Goal
Continue a Scout chat by sending only a previous id and the new message (the pattern behind Gemini's
`previous_interaction_id` and OpenAI's `previous_response_id`), tell a truly server-side chat from a
client-side one, and decide when to use it given retention (Gemini: 55 days paid, 1 day free) and `store=False`.

## The idea
In lesson 08 your code kept the history and resent it every turn. Two vendors offer another way: the
provider **stores** each turn and gives you an id. Next turn you send just the new message plus that id,
and the provider rebuilds the history on its side. This is **server-side state**. It saves you
bookkeeping, but not tokens: the model still reads the whole chain. And your users' words now sit on
the provider's disks for a **retention** period, unless you opt out with `store=False`, which also
switches the feature off.

Analogy: a coat check. You hand over the coat (the chat) and keep a ticket (the id). Convenient, until
the cloakroom closes for the night and your coat is gone.

## Picture
```mermaid
flowchart LR
    subgraph C["client-side (lesson 08)"]
        A1["your code: full history"] -->|"all turns"| M1["model"]
    end
    subgraph S["server-side (this lesson)"]
        A2["your code: last id"] -->|"id + new message"| P["provider store<br/>(kept N days)"]
        P -->|"rebuilt history"| M2["model"]
    end
```

## Step by step
### 1. Start the recorder
Every model call is recorded once to `../data/llm_cassettes/09_server-side-state.yaml` and replayed after that.

In [1]:
import sys; sys.path.insert(0, "..")
import llm_replay
llm_replay.start("09_server-side-state")

LLM mode: replay (09_server-side-state.yaml)


### 2. Setup from lesson 08
The notes, the client, Scout's system text and the `turn` helper. Nothing new.

In [2]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

NOTES = [
    {"id": 1, "text": "Mars has two small moons, Phobos and Deimos."},
    {"id": 2, "text": "Venus spins backwards compared with most planets."},
    {"id": 3, "text": "A day on Mars lasts about 24 hours and 39 minutes."},
    {"id": 4, "text": "Jupiter is the largest planet in the solar system."},
    {"id": 5, "text": "The rover Perseverance landed on Mars in February 2021."},
    {"id": 6, "text": "Saturn's rings are mostly made of ice."},
]
load_dotenv("../.env")
MODEL = os.environ.get("TUT_LLM_MODEL", "gemini-3.5-flash-lite")
client = genai.Client(vertexai=True, api_key=os.environ.get("GOOGLE_CLOUD_API_KEY") or "replay-no-key-needed")
NO_AFC = types.AutomaticFunctionCallingConfig(disable=True)
NOTES_TEXT = "\n".join(f"Note {n['id']}: {n['text']}" for n in NOTES)
SYSTEM_TEXT = ("You are Scout, a research assistant. Answer only from the user's notes below and this chat. "
               "If the notes do not cover it, say so in one sentence. Keep answers to one sentence.\n\n" + NOTES_TEXT)
SCOUT_CONFIG = types.GenerateContentConfig(
    system_instruction=SYSTEM_TEXT, max_output_tokens=60, automatic_function_calling=NO_AFC)

def turn(role: str, text: str) -> types.Content:
    return types.Content(role=role, parts=[types.Part(text=text)])
print(len(NOTES), "notes; model:", MODEL)

6 notes; model: gemini-3.5-flash-lite


### 3. `client.chats` looks like a session. Is it server-side?
The SDK has a chat object: create it once, then call `send_message` with only the new text. That
feels like the provider is remembering. Let's watch the input token count.

In [3]:
chat = client.chats.create(model=MODEL, config=SCOUT_CONFIG)
for text in ["Hi, I'm Asha. I'm researching Mars rovers.", "Which note is about a rover?",
             "What is my name, and what am I researching?"]:
    resp = chat.send_message(text)
    print(f"[{resp.usage_metadata.prompt_token_count:>3} in] {text} -> {resp.text.strip()}")
print("turns held by the chat object:", len(chat.get_history()))

[156 in] Hi, I'm Asha. I'm researching Mars rovers. -> The notes do not mention when the rover Curiosity landed.
[174 in] Which note is about a rover? -> Note 5 is about a rover.
[193 in] What is my name, and what am I researching? -> Your name is Asha, and you are researching Mars rovers.
turns held by the chat object: 6


It remembered the name, and input tokens grew every turn, exactly like lesson 08's `ask`. Where does
the history live? Read the SDK's own source:

In [4]:
import inspect
from google.genai import chats
source = inspect.getsource(chats.Chat.send_message)
print([line.strip() for line in source.splitlines() if "_curated_history +" in line])

['contents_to_model = self._curated_history + [user_input]  # type: ignore[arg-type]']


`client.chats` keeps the history **in your Python process** and resends it. It is client-side state with
a nicer interface: restart the process and the chat is gone. On Vertex AI with an API key (our setup),
`generate_content` and `chats` are all there is; Gemini's server-side feature lives in the Interactions API.

### 4. Real server-side state: read-along code
Gemini Developer API (ai.google.dev), Interactions API. **Not run here:** on Vertex with our key this endpoint
returns 404 (Common mistakes shows it). Facts from the official docs, checked 2026-10-06:
```python
client = genai.Client(api_key=GEMINI_API_KEY)          # Gemini Developer API, not Vertex
turn1 = client.interactions.create(model="gemini-3.8-flash", input="Hi, my name is Phil.")
turn2 = client.interactions.create(model="gemini-3.8-flash", input="What is my name?",
                                   previous_interaction_id=turn1.id)
print(turn2.output_text)
client.interactions.delete(turn1.id)                   # remove it before retention ends
```
`store` defaults to true. `store=False` keeps nothing, and then `previous_interaction_id` cannot point at it.
`tools`, `system_instruction` and `generation_config` are per interaction: send them again each turn.

OpenAI's Responses API has the same shape (read-along, not run: no OpenAI key here):
```python
r1 = client.responses.create(model="gpt-6-astra", input="tell me a joke")
r2 = client.responses.create(model="gpt-6-astra", previous_response_id=r1.id,
                             input=[{"role": "user", "content": "explain why this is funny."}])
conv = client.conversations.create()                   # or: a long-lived conversation object
r3 = client.responses.create(model="gpt-6-astra", conversation=conv.id, input="hi")
```
Responses are stored by default for 30 days (`store=False` opts out); `conversation` and
`previous_response_id` cannot be combined; `instructions` are not carried over to the next response.

### 5. A toy provider store, so we can watch the mechanism
We cannot call those endpoints, so we build the provider's side ourselves: a small class that saves each
turn under an id, with an expiry date. **This is our own toy, not a vendor API.** A fake clock lets us
jump forward in time.

In [5]:
from datetime import datetime, timedelta

class ToyInteractionStore:
    """Plays the provider: keeps turns by id for `retention_days`. Local stand-in, not a real API."""
    def __init__(self, retention_days: int):
        self.retention = timedelta(days=retention_days)
        self.records: dict[str, dict] = {}
        self.now = datetime(2026, 10, 6, 9, 0)            # fake clock
        self.count = 0

    def history(self, interaction_id: str | None) -> list[types.Content]:
        turns = []
        while interaction_id:                              # walk the chain back to the first turn
            rec = self.records.get(interaction_id)
            if rec is None or self.now > rec["expires"]:
                raise LookupError(f"interaction {interaction_id!r} not found (never stored, deleted or expired)")
            turns = [turn("user", rec["input"]), turn("model", rec["output"])] + turns
            interaction_id = rec["previous_id"]
        return turns

`create` mimics `interactions.create`: rebuild the history from the id, call the real model, save the new
turn (only if `store=True`), return its id. Like the real API, the system instruction is per call.

In [6]:
def create(provider: ToyInteractionStore, input: str, previous_interaction_id: str | None = None,
           system_instruction: str | None = None, store: bool = True):
    contents = provider.history(previous_interaction_id) + [turn("user", input)]
    cfg = types.GenerateContentConfig(system_instruction=system_instruction, max_output_tokens=60,
                                      automatic_function_calling=NO_AFC)
    resp = client.models.generate_content(model=MODEL, contents=contents, config=cfg)
    provider.count += 1
    new_id = f"int_{provider.count:03d}"
    if store:
        provider.records[new_id] = {"previous_id": previous_interaction_id, "input": input,
                                    "output": resp.text.strip(), "created": provider.now,
                                    "expires": provider.now + provider.retention}
    print(f"{new_id} [{resp.usage_metadata.prompt_token_count:>3} in] {input} -> {resp.text.strip()}")
    return new_id

### 6. Scout's chat, history kept "by the provider"
Our code now holds one string: the last id. Each call passes it on.

In [7]:
paid = ToyInteractionStore(retention_days=55)
last = create(paid, "Hi, I'm Asha. I'm researching Mars rovers.", system_instruction=SYSTEM_TEXT)
last = create(paid, "Which note is about a rover?", last, system_instruction=SYSTEM_TEXT)
last = create(paid, "What is my name, and what am I researching?", last, system_instruction=SYSTEM_TEXT)
print("our code holds only:", last)

int_001 [156 in] Hi, I'm Asha. I'm researching Mars rovers. -> Hello Asha, the rover Perseverance landed on Mars in February 2021.
int_002 [180 in] Which note is about a rover? -> Note 5 is about a rover.
int_003 [199 in] What is my name, and what am I researching? -> Your name is Asha, and you are researching Mars rovers.
our code holds only: int_003


Scout knows the name, yet our code sent no history. Look at the input counts: they still grow. The
provider rebuilt the history and the model read all of it. OpenAI's docs say so for the real thing:
"all previous input tokens for responses in the chain are billed as input tokens."

### 7. What is stored, and until when
Here is what the provider now holds about Asha.

In [8]:
for rid, rec in paid.records.items():
    print(rid, "<-", rec["previous_id"], "| expires", rec["expires"].date(), "|", rec["input"][:40])

int_001 <- None | expires 2026-11-30 | Hi, I'm Asha. I'm researching Mars rover
int_002 <- int_001 | expires 2026-11-30 | Which note is about a rover?
int_003 <- int_002 | expires 2026-11-30 | What is my name, and what am I researchi


Every user message and every reply, word for word, kept for 55 days. On Gemini's free tier the same
data is kept for 1 day. Paid users can shorten it to 7, 14 or 28 days in AI Studio, or delete an
interaction by id.

### 8. The cloakroom closes: retention runs out
A free-tier user chats today and comes back two days later.

In [9]:
free = ToyInteractionStore(retention_days=1)
monday = create(free, "Hi, I'm Asha. Remember that I study rovers.", system_instruction=SYSTEM_TEXT)
free.now += timedelta(days=2)                              # two days pass
try:
    create(free, "What do I study?", monday, system_instruction=SYSTEM_TEXT)
except LookupError as err:
    print("cannot continue:", err)

int_001 [155 in] Hi, I'm Asha. Remember that I study rovers. -> Hello Asha, I will remember that you study rovers.
cannot continue: interaction 'int_001' not found (never stored, deleted or expired)


The id is now useless. If you kept no copy, the conversation is lost. No model call was made: the
lookup failed first.

### 9. The system instruction is not carried over
Both vendors' docs say instructions belong to one call. Continue Asha's chain but forget to send
`system_instruction`, and ask something the notes do not cover.

In [10]:
_ = create(paid, "What is the capital of France?", last, system_instruction=SYSTEM_TEXT)
_ = create(paid, "What is the capital of France?", last)   # same chain, rules forgotten

int_004 [219 in] What is the capital of France? -> The provided notes do not cover the capital of France.
int_005 [ 78 in] What is the capital of France? -> The capital of France is Paris.


With the rules, Scout stays inside the notes. Without them, the stored chain is still there but the rule
"only from the notes" is gone, and the model answers from general knowledge. The input count also fell
sharply: the notes travelled inside the system instruction, so they vanished too. The history is stored; the rules are not.

### 10. Which one should Scout use?
| | client-side history (lesson 08) | server-side state |
|---|---|---|
| your code stores | every turn | one id |
| input tokens billed | whole history | whole history |
| who keeps users' text | you | provider, 55 d / 1 d (Gemini), 30 d (OpenAI) |
| survives a restart | if you save it | until retention ends |
| works on Vertex + key | yes | no (Interactions 404) |

Scout keeps its own history: it runs on Vertex, and the notes stay under our control.

## At work
- **Teams with privacy rules keep history themselves** and send `store=False`,
  because a stored chat is user data on someone else's disk. → lesson 50
- **Server-side state does not cut the bill:** the whole chain is billed as input every turn. Teams still log
  input tokens per turn and compact long chats. → lessons 12 and 32
- **Expired ids break returning users.** A chat chained only by id dies when retention ends (1 day on
  Gemini's free tier), so apps keep their own copy and fall back to resending it. → lesson 50
- **Instructions and tools must be resent on every chained call;** forgetting them silently drops the
  guardrails, as step 9 showed. → lesson 51

## What just happened
- `client.chats` remembered Asha's name, but its source shows `self._curated_history + [user_input]`: client-side history with a nicer interface.
- Gemini Interactions (`previous_interaction_id`) and OpenAI Responses (`previous_response_id`, `conversation`) store turns on the provider; we read their code, not run it.
- A toy provider store chained Scout's turns by id; our code held one string, yet input tokens still grew.
- The store showed every word Asha typed, with an expiry date; on a 1-day retention the chain broke after two days.
- A chained call without `system_instruction` lost Scout's rules: the history is stored, the rules are not.

## Common mistakes
**1. Calling the Interactions API on Vertex AI.** It is a Gemini Developer API feature. Our Vertex
express-mode client gets a 404.

In [11]:
client.interactions.create(model=MODEL, input="Hi, I'm Asha.")

NotFoundError: <!DOCTYPE html>
<html lang=en>
  <meta charset=utf-8>
  <meta name=viewport content="initial-scale=1, minimum-scale=1, width=device-width">
  <title>Error 404 (Not Found)!!1</title>
  <style>
    *{margin:0;padding:0}html,code{font:15px/22px arial,sans-serif}html{background:#fff;color:#222;padding:15px}body{margin:7% auto 0;max-width:390px;min-height:180px;padding:30px 0 15px}* > body{background:url(//www.google.com/images/errors/robot.png) 100% 5px no-repeat;padding-right:205px}p{margin:11px 0 22px;overflow:hidden}ins{color:#777;text-decoration:none}a img{border:0}@media screen and (max-width:772px){body{background:none;margin-top:0;max-width:none;padding-right:0}}#logo{background:url(//www.google.com/images/branding/googlelogo/1x/googlelogo_color_150x54dp.png) no-repeat;margin-left:-5px}@media only screen and (min-resolution:192dpi){#logo{background:url(//www.google.com/images/branding/googlelogo/2x/googlelogo_color_150x54dp.png) no-repeat 0% 0%/100% 100%;-moz-border-i

Fix: on Vertex, keep history client-side (lesson 08, or `client.chats`). For `previous_interaction_id`,
use a Gemini Developer API client, `genai.Client(api_key=GEMINI_API_KEY)`.

**2. Chaining from a turn sent with `store=False`.** Nothing was saved, so there is nothing to point at.
Our toy behaves like the real API here: the first call works, the second cannot find its parent.

In [12]:
unsaved = create(paid, "Remember: my favourite moon is Phobos.", last, system_instruction=SYSTEM_TEXT, store=False)
create(paid, "Which moon is my favourite?", unsaved, system_instruction=SYSTEM_TEXT)

int_006 [221 in] Remember: my favourite moon is Phobos. -> My notes do not cover your favourite moon.


LookupError: interaction 'int_006' not found (never stored, deleted or expired)

Fix: with `store=False`, resend the history yourself every turn, exactly as lesson 08 did.

## Try it
Real APIs let you delete a stored turn (`client.interactions.delete(id)`). Add `delete(store, interaction_id)`
that removes one record. Start a two-turn chat in a new `ToyInteractionStore(retention_days=55)`, delete the
**first** turn, then try to continue from the **second**. Does it work? Explain why from `history`.

In [13]:
# your code here

Finally, stop the recorder (in record mode this saves the recording).

In [14]:
llm_replay.stop()

## Key terms
- **server-side state** — the provider stores the conversation and you continue it by sending an id plus the new message, instead of the whole history.
- **previous_interaction_id** — the Gemini Interactions API parameter that continues from a stored interaction; OpenAI's Responses API equivalent is `previous_response_id`.
- **data retention** — how long a provider keeps stored requests and replies (Gemini interactions: 55 days paid, 1 day free; OpenAI responses: 30 days).
- **store=False** — the request flag that tells the provider not to keep the turn; it also means you cannot chain from it.
- **client-side history** — your code keeps every turn and resends it each call (lesson 08; also what `client.chats` does).